# P300 speller (calibration mode)

- Target letter is known each time, so every falsh can be labelled as target or non-target

## Packages

- source .speller/bin/activate
- deactivate

- conda activate speller
- conda install -c conda-forge psychopy -y

- python 3.9 but also 3.10 works
- psychopy - conda install -c conda-forge "pyglet<2" -y
- pylsl

## LSL + Faces

- sequence is 12 flashses total 
- 1 row, 1 column falsh is target
- oher 10 are non-target

Markers
* **`experiment_start` / `experiment_end`**: mark the beginning and end of the whole speller session
* **`focus_start/<char>` / `focus_end/<char>`**: mark the period where the participant is shown which character to focus on
* **`target_start/<char>` / `target_end/<char>`**: mark the start and end of one target-character block
* **`sequence_start/<char>/<seq>` / `sequence_end/<char>/<seq>`**: mark the start and end of one full shuffled row+column flashing sequence
* **`flash_on/<kind>/<idx>/target_<0 or 1>/char_<char>/seq_<seq>/flash_<flash>`**: marks the exact onset of a flashed row or column, used to time-lock EEG epochs for P300 analysis
* **`flash_off/<kind>/<idx>/target_<0 or 1>/char_<char>/seq_<seq>/flash_<flash>`**: marks the end of that flash, mainly useful for timing checks and debugging
* **`target_1` vs `target_0`**: indicates whether the flashed row/column contains the attended target character, giving the label for CNN training
* **`kind` and `idx`**: specify which row or column was flashed, so predictions can later be combined into the final selected character



Full pipeline needed: 
1. Speller
- sends marker stream via LSL
2. EEG acquisition software 
- sends EEG stream via LSL
3. Recorder
- records EEG + markers into XDF
4. Processing
- reads the EEG marker timing, epochs the data, trains / runs CNN

In [ ]:
from psychopy import visual, core, event
import random
import os

# LSL imports ---------------------
# StreamInfo describes the stream, StreamOutlet sends markers out
from pylsl import StreamInfo, StreamOutlet


# Parameters ---------------------
GRID_ROWS = 6
GRID_COLS = 6

# Flash timing (seconds)
FLASH_DUR = 0.100   # how long a row/col stays highlighted
ISI       = 0.075   # pause between flashes (inter-stimulus interval)

N_SEQS    = 10      # how many full row+col sequences per target letter (6×6 grid = one sequence is 6 row flashes + 6 column flashes)

# Extra timing
FOCUS_DUR = 1.0     # how long the focus cue screen stays on
REST_DUR  = 0.8     # short relax screen after each target block

# Letter appearance
TEXT_H_NORMAL = 40
TEXT_H_HL     = 40  # can make bigger if “pop” effect is wanted

# Symbols in the 6x6 grid (must be exactly 36)
SYMS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890")
assert len(SYMS) == GRID_ROWS * GRID_COLS

# Demo target word (each char becomes a separate calibration-like block)
word = "PLAY"


# Cell images (cell_00.jpg ... cell_35.jpg) in repo ./Images ---------------------
IMAGE_DIR = "Images"
CELL_IMAGES = [os.path.join(IMAGE_DIR, f"cell_{i:02d}.jpg") for i in range(GRID_ROWS * GRID_COLS)]
# Mapping rule: grid index 0..35 uses Images/cell_00.jpg .. Images/cell_35.jpg


# Window + clock (BLACK background) ---------------------
# color=[-1,-1,-1] is black in PsychoPy's -1..+1 color space
win = visual.Window(size=(1000, 700), units="pix", fullscr=False, color=[-1, -1, -1])
exp_clock = core.Clock()  # used for timestamps in event_log


# LSL marker outlet ---------------------
# This creates one LSL stream for event markers.
# channel_count=1 means each marker is a single value
# nominal_srate=0 means markers are irregular events, not continuous samples
# channel_format="string" means markers are text labels like "flash_on/row/2/..."
marker_info = StreamInfo(
    name="P300Markers",
    type="Markers",
    channel_count=1,
    nominal_srate=0,
    channel_format="string",
    source_id="p300_speller_markers_v1"
)
marker_outlet = StreamOutlet(marker_info)


def send_lsl(marker):
    # Sends one marker sample into the LSL stream
    marker_outlet.push_sample([marker])


# Quit handling ---------------------
should_quit = False

def quit_now():
    # soft stop: set flag, then main loop will close window cleanly
    global should_quit
    should_quit = True

def check_quit():
    # call frequently so ESC/Q quits quickly even during flashes
    keys = event.getKeys()
    if "escape" in keys or "q" in keys:
        quit_now()


# Rectangle using ShapeStim (visual.Rect crashed) ---------------------
def make_rect(win, width, height, pos=(0, 0), fillColor=None, lineColor=None, lineWidth=1):
    # unit square scaled by "size" to desired width/height
    verts = [(-0.5, 0.5), (0.5, 0.5), (0.5, -0.5), (-0.5, -0.5)]
    return visual.ShapeStim(
        win,
        vertices=verts,
        closeShape=True,
        size=(width, height),
        pos=pos,
        fillColor=fillColor,
        lineColor=lineColor,
        lineWidth=lineWidth,
        units="pix",
        interpolate=True,
        autoLog=False,
    )


# Layout ---------------------
# Background panel behind the grid (gives contrast vs black window)
panel = make_rect(
    win,
    width=900,
    height=640,
    pos=(0, 0),
    fillColor=[-0.9, -0.9, -0.9],
    lineColor=[-0.4, -0.4, -0.4],
    lineWidth=2,
)

title = visual.TextStim(
    win, text="P300 Speller", pos=(0, 335), height=32,
    color="white", bold=True, font="Arial"
)

cue = visual.TextStim(
    win, text="", pos=(0, -270), height=28,
    color="white", bold=True, font="Arial"
)

# instructions
instr = visual.TextStim(
    win, text="Press SPACE to start. ESC/Q to quit.", pos=(0, -305),
    height=22, color="white", font="Arial"
)

status = visual.TextStim(
    win, text="", pos=(0, -330),
    height=18, color="white", font="Arial"
)

# Focus text only
# More negative y -> lower on screen (because y=0 is center)
focus_text = visual.TextStim(
    win, text="", pos=(0, -265), height=28,
    color="white", bold=True, font="Arial"
)


# Grid geometry ---------------------
# These numbers control the grid footprint and spacing
grid_w, grid_h = 720, 450
x0, y0 = -grid_w / 2, 250  # top-left anchor for the grid
dx = grid_w / (GRID_COLS - 1)
dy = grid_h / (GRID_ROWS - 1)


# Build grid: boxes + (hidden) images + letters ---------------------
cell_boxes = []  # each cell border/box
cell_imgs  = []  # per-cell image overlay (faces), normally hidden
cells_text = []  # letter/number label per cell

box_w = dx * 0.92
box_h = dy * 0.92

GRID_LINE = [-0.35, -0.35, -0.35]  # subtle border line on the gray panel

for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        idx = r * GRID_COLS + c  # 0..35 flat index
        x = x0 + c * dx
        y = y0 - r * dy

        # 1) the box outline for the cell
        cell_boxes.append(
            make_rect(
                win,
                width=box_w,
                height=box_h,
                pos=(x, y),
                fillColor=None,       # no fill by default
                lineColor=GRID_LINE,
                lineWidth=1,
            )
        )

        # 2) the face image (hidden unless the row/col is flashing)
        cell_imgs.append(
            visual.ImageStim(
                win,
                image=CELL_IMAGES[idx],
                pos=(x, y),
                size=(box_w * 0.90, box_h * 0.90),
                opacity=0.0,          # hidden normally
                interpolate=True,
                autoLog=False
            )
        )

        # 3) the symbol text shown in the cell (hidden during flash)
        cells_text.append(
            visual.TextStim(
                win,
                text=SYMS[idx],
                pos=(x, y),
                height=TEXT_H_NORMAL,
                color="white",
                bold=True,
                font="Arial",
                opacity=1.0
            )
        )


# Highlight colors (optional: box glow (if needed)) ---------------------
# Used to indicate which row/col is currently flashing and the focus cell
HL_FILL = [-0.15, 0.35, 0.55]
HL_EDGE = [0.20, 0.85, 1.00]


# Logging -----------------------
event_log = []

def log_event(ev_type, **fields):
    # Internal log only, useful for debugging even when LSL is running
    event_log.append({"t": exp_clock.getTime(), "type": ev_type, **fields})


# Drawing + flashing -----------------------
def _idxs_for_highlight(kind, k):
    # Converts a row index or column index into the list of cell indices to highlight
    if kind == "row":
        return [k * GRID_COLS + c for c in range(GRID_COLS)]
    if kind == "col":
        return [r * GRID_COLS + k for r in range(GRID_ROWS)]
    return []

def draw_grid(highlight=None, hide_letters_on_flash=True):
    # Draws the full scene in the correct draw order
    panel.draw()
    title.draw()

    # Reset visuals every frame (so there is no “sticky” highlight)
    for b in cell_boxes:
        b.fillColor = None
        b.lineColor = GRID_LINE

    for im in cell_imgs:
        im.opacity = 0.0

    for t in cells_text:
        t.height = TEXT_H_NORMAL
        t.color = "white"
        t.opacity = 1.0

    # Apply highlight if a row/col is flashing
    if highlight is not None:
        kind, k = highlight
        idxs = _idxs_for_highlight(kind, k)
        for i in idxs:
            # glow box
            cell_boxes[i].fillColor = HL_FILL
            cell_boxes[i].lineColor = HL_EDGE

            # show face during flash
            cell_imgs[i].opacity = 1.0

            # optional text size bump
            cells_text[i].height = TEXT_H_HL

            # hide the letter while the face is shown
            if hide_letters_on_flash:
                cells_text[i].opacity = 0.0

    # Draw order matters: box -> image -> text
    for b in cell_boxes:
        b.draw()
    for im in cell_imgs:
        if im.opacity > 0:
            im.draw()
    for t in cells_text:
        if t.opacity > 0:
            t.draw()

    cue.draw()
    instr.draw()
    status.draw()

def draw_focus_screen(target_char):
    # Before flashing begins, show which symbol to focus on
    # Here: we show the target cell's face image inside the grid and hide its letter
    panel.draw()
    title.draw()

    for b in cell_boxes:
        b.fillColor = None
        b.lineColor = GRID_LINE

    for im in cell_imgs:
        im.opacity = 0.0

    for t in cells_text:
        t.height = TEXT_H_NORMAL
        t.color = "white"
        t.opacity = 1.0

    target_idx = SYMS.index(target_char)

    # highlight target cell outline/fill
    cell_boxes[target_idx].fillColor = HL_FILL
    cell_boxes[target_idx].lineColor = HL_EDGE

    # show only the face in target cell, hide its letter
    cell_imgs[target_idx].opacity = 1.0
    cells_text[target_idx].opacity = 0.0

    # draw grid
    for b in cell_boxes:
        b.draw()
    if cell_imgs[target_idx].opacity > 0:
        cell_imgs[target_idx].draw()
    for t in cells_text:
        if t.opacity > 0:
            t.draw()

    # draw the instruction text
    focus_text.text = f"Focus on: {target_char}"
    focus_text.draw()

    instr.draw()
    status.draw()

def flash(kind, idx, tr, tc, target_char, seq_i, flash_i):
    # One flash: highlight a single row or a single column
    check_quit()
    if should_quit:
        return

    # "target flash" means either the row of the target symbol OR the column of the target symbol
    is_target = (kind == "row" and idx == tr) or (kind == "col" and idx == tc)

    # flash on
    # IMPORTANT:
    # We schedule the LSL marker on the same screen refresh with callOnFlip().
    # That way the marker is aligned as closely as possible with the actual visual onset.
    draw_grid(highlight=(kind, idx), hide_letters_on_flash=True)

    # Added seq_i and flash_i so each flash is easier to identify later in the receiver/CNN code
    marker_on = f"flash_on/{kind}/{idx}/target_{int(is_target)}/char_{target_char}/seq_{seq_i}/flash_{flash_i}"
    win.callOnFlip(send_lsl, marker_on)
    win.callOnFlip(log_event, "flash_on", kind=kind, idx=idx, is_target=int(is_target), target_char=target_char, seq=seq_i, flash=flash_i)

    win.flip()
    core.wait(FLASH_DUR)

    check_quit()
    if should_quit:
        return

    # flash off
    # Also sending an offset marker can be useful for debugging or timing checks
    draw_grid(highlight=None)

    marker_off = f"flash_off/{kind}/{idx}/target_{int(is_target)}/char_{target_char}/seq_{seq_i}/flash_{flash_i}"
    win.callOnFlip(send_lsl, marker_off)
    win.callOnFlip(log_event, "flash_off", kind=kind, idx=idx, is_target=int(is_target), target_char=target_char, seq=seq_i, flash=flash_i)

    win.flip()
    core.wait(ISI)

def run_target(target_char):
    # Runs N_SEQS sequences for one target symbol
    if target_char not in SYMS:
        raise ValueError(f"Target '{target_char}' not in grid symbols.")

    # Convert the target symbol into its row/col indices
    target_idx = SYMS.index(target_char)
    tr = target_idx // GRID_COLS
    tc = target_idx % GRID_COLS

    cue.text = ""
    status.text = ""

    # Focus cue screen (1 second)
    # Added focus_start/focus_end markers so you can separate cue period from actual flashing period
    draw_focus_screen(target_char)
    win.callOnFlip(send_lsl, f"focus_start/{target_char}")
    win.callOnFlip(log_event, "focus_start", target_char=target_char, tr=tr, tc=tc)
    win.flip()
    core.wait(FOCUS_DUR)

    check_quit()
    if should_quit:
        return

    draw_focus_screen(target_char)
    win.callOnFlip(send_lsl, f"focus_end/{target_char}")
    win.callOnFlip(log_event, "focus_end", target_char=target_char)
    win.flip()

    # Marker saying a new target block has started
    log_event("target_start", target_char=target_char, target_idx=target_idx, tr=tr, tc=tc)
    send_lsl(f"target_start/{target_char}")

    for seq_i in range(N_SEQS):
        check_quit()
        if should_quit:
            return

        # One sequence (shuffled)
        order = [("row", r) for r in range(GRID_ROWS)] + [("col", c) for c in range(GRID_COLS)]
        random.shuffle(order)

        # Marker saying one new 12-flash sequence has started
        log_event("sequence_start", target_char=target_char, seq=seq_i)
        send_lsl(f"sequence_start/{target_char}/{seq_i}")

        for flash_i, (kind, idx) in enumerate(order):
            flash(kind, idx, tr=tr, tc=tc, target_char=target_char, seq_i=seq_i, flash_i=flash_i)
            if should_quit:
                return

        # Marker saying this sequence has ended
        log_event("sequence_end", target_char=target_char, seq=seq_i)
        send_lsl(f"sequence_end/{target_char}/{seq_i}")

    # short rest screen
    cue.text = "Relax..."
    draw_grid(None)
    win.flip()
    core.wait(REST_DUR)

    # Marker saying the current target block is finished
    log_event("target_end", target_char=target_char)
    send_lsl(f"target_end/{target_char}")


# Main -----------------------
try:
    cue.text = ""
    status.text = ""
    draw_grid(None)
    win.flip()

    # Wait for SPACE to start (or ESC/Q to quit)
    while True:
        keys = event.getKeys()
        if "space" in keys:
            break
        if "escape" in keys or "q" in keys:
            quit_now()
            break
        core.wait(0.05)

    if not should_quit:
        # Marker saying the whole experiment has started
        log_event("experiment_start")
        send_lsl("experiment_start")

        for ch in word:
            run_target(ch)
            check_quit()
            if should_quit:
                break

        # Marker saying the whole experiment has ended
        log_event("experiment_end")
        send_lsl("experiment_end")
        core.wait(0.2)

        cue.text = "Calibration phase ended"
        status.text = "Press ESC/Q or close the window."
        instr.text = ""
        draw_grid(None)
        win.flip()

        # Keep window open until quit
        while not should_quit:
            check_quit()
            core.wait(0.05)

finally:
    try:
        win.close()
    except Exception:
        pass

print(f"Logged {len(event_log)} events.")
print("First 10 events:")
for row in event_log[:10]:
    print(row)